# Pick and place a cube with a Franka Panda in the apartment (Isaac Sim, real-robot mode)

The same task as `stretch_pick_place_cram.ipynb` — grasp a cube off one stand and
put it on another — with the hard parts of the Stretch taken out. No mobile base
to drive, no telescoping arm, and a hand whose fingers slide straight at each
other instead of swinging on an arc. What is left is the grasp itself.

As in the Stretch demo, **the cube is a real rigid body in Isaac Sim**. The
fingers have to close on it for real and friction has to hold it. The sim
publishes the cube's true pose alongside, so `cube_status()` can print what CRAM
*believes* against what the physics actually did; a grasp that misses shows up
as the two drifting apart.

**Kernel**: select **CRAM**.

## Scene

The apartment, with the Panda mounted at working height inside it and two stands
within its reach. Everything below is in the giskard `map` frame.

| | map |
|---|---|
| Panda base | (1.14, −0.30, 0.933), yaw 180° |
| pick position | (0.69, −0.10) |
| place position | (0.69, −0.50) |

The base pose is given in the apartment USD's own frame
(`PANDA_BASE_POSITION_IN_USD = (7.14, −5.30, 0.933)`); x and y are composed with
the prim placement `USD_PRIM_POSITION_IN_MAP = (−6.0, 5.0, 0.0701)` to land in
`map`, while **z is taken verbatim** — the prim's 7 cm lift belongs to the
apartment geometry, not to the table the arm is bolted to. Both the Isaac side
and the giskard world config read those same constants, so the arm giskard plans
for and the arm Isaac renders are in the same place by construction.

The pick and place positions are written as offsets in the **robot's own frame**
(`PANDA_REACH_OFFSETS`, 0.45 m ahead and ±0.2 m to the side) and rotated into
`map`, so they stay in front of the arm if the mounting pose ever changes. Both
are 0.49 m from the base, well inside the 0.85 m reach.

**There are no pedestals.** The arm is bolted to a table and works on that table,
so the cube is released just above it (`drop_height`, 3 cm) and physics settles it
onto the real surface. `spawn_props` prints the pose it actually settled at, which
is the only way to learn that surface's true height — the table comes from the
apartment USD and is not modelled in the URDF giskard plans against.

## How the gripper is told to come at the cube

CRAM's grasp maths assumes a tool frame whose **x-axis is the approach axis**.
The Panda's hand points its **z-axis** out between the fingers, so
`PandaGripper.front_facing_orientation` carries the 90° rotation about y that
reconciles the two (see `cram_vrb_lab/robots/panda/semantic_model.py`). With that
in place the standard grasp descriptions mean what they say, and

    ApproachDirection.FRONT + VerticalAlignment.TOP

is a straight top-down grasp: the hand descends along map −z with the fingers
closing along map y. That is the natural grasp for something standing on a table
and the one used below.

## Start the simulation and giskard server

Separate scripts from the Stretch demos — a different robot and a different set
of topics — so both can coexist even though they now share the apartment. The
viewport is framed on the arm's workspace rather than the apartment as a whole,
since the props are 5 cm objects.

Startup is slower than the Stretch demos': giskard parses the apartment URDF into
its collision world on top of the robot.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

# import os
# os.environ["DISPLAY"] = ":0"

from launcher import (
    PANDA_SERVER_SCRIPT,
    PANDA_SIM_MARKER,
    PANDA_SIM_SCRIPT,
    start_giskard_server,
    start_isaac_sim,
    start_rviz,
    stop,
)
rviz_proc = start_rviz()
sim_proc = start_isaac_sim(sim_script=PANDA_SIM_SCRIPT, marker=PANDA_SIM_MARKER)
giskard_proc = start_giskard_server(server_script=PANDA_SERVER_SCRIPT)

killed a stale instance of rviz2
starting rviz2, logging to /tmp/rviz.log
starting isaac sim, logging to /tmp/isaac_sim.log
isaac sim ready after 22s
starting giskard server, logging to /tmp/giskard_server.log
giskard server ready after 14s


## Connect and build a CRAM `Context`

Same as the Stretch notebooks: fetch the world the giskard server built over its
`fetch_world` service, keep it live with a `WorldSynchronizer`, and wrap it in a
`Context`.

The Stretch's `alternative_motion_mappings` do not apply — they exist to give a
mobile base better motions, and there is no base here; CRAM's default
`MoveToolCenterPointMotion`, a plain `CartesianPose` on the tool frame, is
exactly right. The Panda contributes one mapping of its own,
`PandaMoveGripper`, which is what stops a grasp from hanging forever: see step 3.

In [2]:
import threading

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

from cram_vrb_lab.robots.panda.motions import PANDA_MOTION_MAPPINGS
from cram_vrb_lab.robots.panda.semantic_model import Panda

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_panda_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Panda)
robot = robot[0] if robot else Panda.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=PANDA_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

connected, robot: Panda


## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()`
builds each action's giskard motion and streams it to the running server.

In [3]:
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import execute_single, sequential


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')

## 1. Put the props into the digital twin

Isaac already has the cube as a physics body. CRAM plans against
the twin, so it has to exist there too — `add_props_to_twin`
builds it from the shared `PANDA_APARTMENT_LAYOUT`. The change goes out over
`/world_sync`, so the giskard server's copy of the world learns about it. The
apartment is already in that world: the giskard server merges it in at startup,
so collision avoidance covers the room as well.

With no pedestals in the layout only the cube is added. Whatever it rests on
belongs to the scene — and if that table is not in the apartment URDF either,
giskard plans as though nothing were underneath it.

`sync_cube_from_sim` then snaps the twin's cube onto the pose Isaac's physics
actually settled it at — a one-shot perception stand-in, and the only channel
through which the twin ever learns the cube is not where CRAM assumed.

In [4]:
import numpy as np

from cram_vrb_lab.scenes.props import constants as props
from cram_vrb_lab.scenes.props.constants import PANDA_APARTMENT_LAYOUT as PANDA_LAYOUT
from cram_vrb_lab.scenes.props.twin_props import (
    CubePoseSensor,
    add_props_to_twin,
    sync_cube_from_sim,
)

cube = add_props_to_twin(world, layout=PANDA_LAYOUT)
cube_sensor = CubePoseSensor(node)

print('twin cube:', np.round(np.asarray(cube.global_pose.to_np())[:3, 3], 3))
print('sim cube: ', np.round(sync_cube_from_sim(world, cube, cube_sensor), 3))

twin cube: [ 0.69  -0.1    0.988]
sim cube:  [ 0.69  -0.1    0.957]


In [ ]:
# from semantic_digital_twin.adapters.mesh import STLParser
# from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk
# from semantic_digital_twin.world_description.connections import FixedConnection
# from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix

# MILK_STL = str(
#     REPO
#     / "cognitive_robot_abstract_machine"
#     / "coraplex"
#     / "resources"
#     / "objects"
#     / "milk.stl"
# )

# if world.get_semantic_annotations_by_type(Milk):
#     print("milk already in the world")
# else:
#     milk_world = STLParser(MILK_STL).parse()
#     with world.modify_world():
#         world.merge_world(
#             milk_world,
#             FixedConnection(
#                 parent=world.get_body_by_name("apartment_root"),
#                 child=milk_world.root,
#                 parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
#                    6.85, -4.95, 0.9617
#                 ),
#             ),
#         )
#         world.add_semantic_annotations([Milk(root=world.get_body_by_name("milk.stl"))])
# print("milk spawned in", drawer_body.name)

### Believed vs. actual

The one measurement this notebook is built around. CRAM's `AttachNode` moves the
cube along with the gripper *in the twin* the moment the close-gripper motion
finishes — whether or not the physical fingers caught anything. So the twin
always reports a successful grasp. Isaac does not.

While the cube is held, a gap of a centimetre or two is normal (the twin freezes
it at the tool frame; the real cube sits wherever the fingers gripped it). A gap
that keeps growing means the cube was left behind or dropped.

In [5]:
def cube_status(label=''):
    """Print where CRAM believes the cube is vs where Isaac's physics has it."""
    believed = np.asarray(cube.global_pose.to_np())[:3, 3].ravel()
    actual = np.array(cube_sensor.position())
    print(f'{label:14s} twin {np.round(believed, 3)}  '
          f'sim {np.round(actual, 3)}  gap {np.linalg.norm(believed - actual):.3f} m')
    return believed, actual


cube_status('start')

start          twin [ 0.69  -0.1    0.957]  sim [ 0.69  -0.1    0.957]  gap 0.000 m


(array([ 0.69000006, -0.09999993,  0.95679677]),
 array([ 0.69000006, -0.09999993,  0.95679677]))

## 2. Park the arm and open the hand

`ParkArmsAction` looks up the Panda's park configuration (Franka's own "ready"
pose) and `SetGripperAction` its open finger travel, both from the semantic
model. Unlike the Stretch, no widening is needed: the hand's `GripperState.OPEN`
is 0.038 m of travel per finger, i.e. **0.076 m between the pads** against a
0.05 m cube — 1.3 cm of clearance either side, which is how much approach error
the grasp tolerates before a finger knocks the cube off its stand.

In [6]:
from coraplex.datastructures.enums import Arms
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

from cram_vrb_lab.robots.panda.joints import FINGER_JOINTS, gripper_pad_gap

# run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
# run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

# travel = world.get_connection_by_name(FINGER_JOINTS[0]).position
# print(f'fingers at {travel:.3f} m travel = {gripper_pad_gap(travel):.3f} m between '
#       f'the pads (needs to clear the {props.CUBE_SIZE:.3f} m cube)')

`Arms.LEFT` is not a claim about handedness — the Panda has one arm, and CRAM's
`ViewManager` hands back the only arm there is whatever you ask for. It just has
to be *some* member of the enum.

## 3. Pick the cube up and put it down

Both halves run as **one plan**, and that is not a stylistic choice — it is the
only way to make the place go top-down.

**`PlaceAction` takes no grasp.** Its fields are just `object_designator`,
`target_location`, `arm`. It recovers the orientation by looking for a
`PickUpAction` *earlier in the same plan* and reusing that pick's
`grasp_description`
(`coraplex/robot_plans/actions/core/placing.py`), falling back to
`FRONT` + `NoAlignment` when it finds none. Run the pick and the place as two
separate `execute_single(...)` plans and the place always gets that fallback —
whose three rotation factors are all identity, so the target tool rotation
reduces to exactly the gripper's `front_facing_orientation`. For the Panda that
is the 90° rotation about y, i.e. hand z along map **+x**: a horizontal approach,
with a horizontal retract that drags the hand across the table.

`PickAndPlaceAction` ("without moving the base") keeps them in one plan, so the
place inherits `FRONT` + `TOP`. It expands to:

    ParkArms → PickUp(grasp) → ParkArms → Place → ParkArms

With `FRONT` + `TOP` every tool pose is vertical, in map coordinates:

| | position (map) | hand points |
|---|---|---|
| pick pre-grasp | (0.69, −0.10, 1.033) | down |
| pick grasp | (0.69, −0.10, 0.958) | down |
| pick lift | (0.69, −0.10, 1.008) | down |
| place descend | (0.69, −0.50, 1.008) | down |
| place release | (0.69, −0.50, 0.958) | down |
| place retract | (0.69, −0.50, 1.033) | down |

(Nominal, from `surface_z = 0.933`; the grasp is re-planned against the cube's
settled pose, so the real numbers follow wherever the table actually is. The
7.5 cm of approach clearance is the cube's half-height plus
`GraspDescription.manipulation_offset`.)

`collision_avoidance=False` for the same reason as before — the hand has to get
*inside* the avoidance margin of the cube and the table, which is exactly what the
margin forbids. Note that one `real_robot(...)` context now wraps the whole
composite, so the parks and the transfer run without external collision avoidance
too, not just the two reaches.

**Why the close-gripper step needs a Panda-specific motion.** CRAM turns a
gripper state into a joint-position goal on the fingers, and under closed-loop
control that goal is checked against the *measured* finger positions. A hand
closing on a rigid object never reaches them — the cube stops the fingers 2.5 cm
short of closed — so the default 1 cm tolerance is never met and the grasp hangs
right there, holding the cube, looking for all the world like it worked.
`PandaMoveGripper` (in `cram_vrb_lab/robots/panda/motions.py`) gives *closing*
a 3 cm tolerance. Opening keeps the tight one; nothing obstructs it, and the
place's release needs it to open fully.

Ending the close is not the same as letting go: the fingers keep driving towards a
target that now lies inside the cube, so they go on squeezing through the lift and
the transfer.

We re-sync from the sim first, so the grasp is planned against where the cube
really is rather than where it was spawned.

In [ ]:
from coraplex.datastructures.enums import ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.robot_plans.actions.composite.transporting import PickAndPlaceAction
from coraplex.view_manager import ViewManager
from semantic_digital_twin.spatial_types import Point3
from semantic_digital_twin.spatial_types.spatial_types import Pose

grasp = GraspDescription(
    ApproachDirection.FRONT,   # with TOP below: approach along map -z
    VerticalAlignment.TOP,
    ViewManager.get_end_effector_view(Arms.LEFT, robot),
)
place_target = Pose(
    Point3.from_iterable((0.69, -0.3, 0.958)),
    reference_frame=world.root,
)

sync_cube_from_sim(world, cube, cube_sensor)
run_plan(
    execute_single(
        # One plan, so PlaceAction can find this grasp on the PickUpAction node.
        PickAndPlaceAction(cube, place_target, Arms.LEFT, grasp_description=grasp),
        context=context,
    ),
    collision_avoidance=False,
)
cube_status('after place')

### What the merge costs

Running the pick and the place as one plan is what buys the top-down place, but it
takes away the measurement in between. Previously `cube_status('after grasp')` ran
while the cube was in the air, which is where a grasp that closes on nothing shows
itself immediately. Now a cube that is lifted and then dropped mid-transfer looks
the same from the outside as one that was never picked up — both end with the cube
somewhere it should not be, and only the verdict below reports it. Watch the Isaac
viewport for the difference, or re-add a `cube_status` call by splitting the
composite back into `sequential([PickUpAction(...), PlaceAction(...)])` and
accepting that you cannot interleave Python between plan nodes.

The composite already ends with a park, so no separate `ParkArmsAction` is needed
after it.

## 4. Verdict

The cube's true resting position against where it was asked to go. Both parts
matter: the height says it is still on the table rather than on the floor, and the
x/y error says it is at the *right* spot on it — a cube that was never picked up
would score a perfect height and be 0.4 m out in y, which is exactly the failure
the merged plan can no longer catch mid-flight.

In [ ]:
_, actual = cube_status('final')
target = np.array(PANDA_LAYOUT.cube_target_position)
error = actual - target
placed = np.linalg.norm(error[:2]) < 0.05 and abs(error[2]) < 0.02
print(f'target {np.round(target, 3)}')
print(f'error  {np.round(error, 3)}   |xy| {np.linalg.norm(error[:2]):.3f} m')
print('PLACED on the table' if placed else 'NOT placed -- see cube_status above')

In [7]:
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
)

drawer_body = world.get_body_by_name("cabinet9_drawer1")
handle_body = world.get_body_by_name("cabinet9_drawer1_handle")

if not world.get_semantic_annotations_by_type(Drawer):
    with world.modify_world():
        world.add_semantic_annotation_recursively(
            Drawer(root=drawer_body, handle=Handle(root=handle_body))
        )
print("drawer annotated:", drawer_body.name, "with handle", handle_body.name)

drawer annotated: cabinet9_drawer1 with handle cabinet9_drawer1_handle


In [8]:
from coraplex.datastructures.enums import MovementType
from coraplex.robot_plans.motions.gripper import MoveToolCenterPointMotion
from semantic_digital_twin.spatial_types import Point3
from semantic_digital_twin.spatial_types.spatial_types import Pose

# Out past the worktop's front edge (x = 1.287), then down beside the cabinet.
# TRANSLATION = position-only goals, so the QP picks the wrist on the way and only
# OpenAction constrains the orientation.
approach = [
    (1.80, 0, 1.02),    # clear of the edge, still above the top face
    # (1.40, 0, 0.715),   # down into the free corridor, at handle height
]
for point in approach:
    run_plan(
        execute_single(
            MoveToolCenterPointMotion(
                Pose(Point3.from_iterable(point), reference_frame=world.root),
                Arms.LEFT,
                movement_type=MovementType.TRANSLATION,
            ),
            context=context,
        ),
        collision_avoidance=False,
    )

[INFO] [1785408061.456761974] [cram_panda_node]: giskard/command Goal #0 accepted


done


[INFO] [1785408089.414502295] [cram_panda_node]: giskard/command Goal #0 result received


In [ ]:
from coraplex.robot_plans.actions.core.container import OpenAction, CloseAction

run_plan(
    execute_single(OpenAction(handle_body, Arms.LEFT), context=context),
    collision_avoidance=True,
)
print("drawer joint:", world.get_connection_by_name("cabinet10_drawer_middle_joint").position)

[INFO] [1785408262.631059069] [cram_panda_node]: giskard/command Goal #0 accepted


: 

## Shutdown

In [ ]:
stop(patterns=(PANDA_SIM_SCRIPT.name, PANDA_SERVER_SCRIPT.name))